# 04 · Error analysis: what does the model still get wrong, and why?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pavanadityav-ai/telugu-asr-whisper/blob/main/notebooks/04_error_analysis.ipynb)

**Goal:** go beyond one number. Compare the baseline with the fine-tuned model, find which Telugu sounds and words cause errors, and label the worst mistakes **as a native speaker**.

**Runtime:** no GPU needed. Needs the results saved by notebooks 01, 02 and 03.

In [ ]:
# Setup: install libraries, get the project code, connect Google Drive
!pip install -q -U datasets jiwer
import os, sys
if not os.path.isdir("telugu-asr-whisper"):
    !git clone -q https://github.com/pavanadityav-ai/telugu-asr-whisper
sys.path.insert(0, "telugu-asr-whisper")

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/telugu-asr-whisper"  # results are saved here
os.makedirs(OUT, exist_ok=True)
print("Saving results to:", OUT)

In [ ]:
import os
import unicodedata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.metrics import cer, char_confusions, wer, word_hits

base = pd.read_csv(f"{OUT}/baseline_whisper-small_test_predictions.csv").fillna("")
ft = pd.read_csv(f"{OUT}/finetuned_whisper-small_test_predictions.csv").fillna("")
both = base.merge(ft, on=["id", "seconds", "reference"], suffixes=("_base", "_ft"))
print(len(both), "test sentences")

## 1. Overall results

In [ ]:
results = pd.DataFrame([
    {"Model": "whisper-small (no fine-tuning)",
     "WER %": 100 * wer(base.reference, base.prediction), "CER %": 100 * cer(base.reference, base.prediction)},
    {"Model": "whisper-small (fine-tuned)",
     "WER %": 100 * wer(ft.reference, ft.prediction), "CER %": 100 * cer(ft.reference, ft.prediction)},
]).round(1)
results

## 2. Sentence by sentence: how many got better?

In [ ]:
change = both.cer_ft - both.cer_base
print(f"Better: {(change < 0).sum()}   Same: {(change == 0).sum()}   Worse: {(change > 0).sum()}")

plt.figure(figsize=(7, 3))
bins = np.linspace(0, 1.5, 31)
plt.hist(both.cer_base.clip(upper=1.5), bins=bins, alpha=0.6, label="baseline")
plt.hist(both.cer_ft.clip(upper=1.5), bins=bins, alpha=0.6, label="fine-tuned")
plt.xlabel("character error rate per sentence"); plt.ylabel("sentences"); plt.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/cer_per_sentence.png", dpi=150); plt.show()

## 3. Are longer clips harder?

In [ ]:
both["length_group"] = pd.qcut(both.seconds, 4, labels=["shortest 25%", "short", "long", "longest 25%"])
both.groupby("length_group", observed=True)[["cer_base", "cer_ft"]].mean().mul(100).round(1)

## 4. Which Telugu characters get confused?
Each row is one kind of mistake: **S** = one character heard as another, **D** = character missing, **I** = extra character.

As a native speaker, look for patterns such as:
short vs long vowels (ి / ీ, ు / ూ), aspirated vs plain consonants (క / ఖ, ద / ధ), dental vs retroflex (త / ట, న / ణ), ల / ళ, and the anusvara ం.

In [ ]:
def char_name(ch):
    return "-" if ch is None else f"{ch}  ({unicodedata.name(ch, '?').replace('TELUGU ', '').lower()})"

conf = char_confusions(ft.reference, ft.prediction)
top = pd.DataFrame([{"type": op, "correct": char_name(r), "model wrote": char_name(h), "count": n}
                    for (op, r, h), n in conf.most_common(20)])
top.to_csv(f"{OUT}/top_char_confusions.csv", index=False)
top

## 5. Words seen in training vs new words
Telugu has many word forms, so many test words never appeared in training. Does the model struggle more with those?

In [ ]:
vocab_file = f"{OUT}/train_vocab.txt"
if os.path.exists(vocab_file):
    train_vocab = set(open(vocab_file, encoding="utf-8").read().split())
else:  # notebook 01 was not run: build the vocabulary here
    from src.audio_utils import load_fleurs_telugu
    from src.text_utils import normalize_text
    train_vocab = {w for t in load_fleurs_telugu("train")["transcription"] for w in normalize_text(t).split()}

rows = []
for name, d in [("baseline", base), ("fine-tuned", ft)]:
    hits = word_hits(d.reference, d.prediction)
    for seen in (True, False):
        group = [ok for w, ok in hits if (w in train_vocab) == seen]
        rows.append({"model": name, "word type": "seen in training" if seen else "never seen",
                     "words": len(group), "recognised correctly %": 100 * np.mean(group)})
pd.DataFrame(rows).round(1)

## 6. Wrong script and English words

In [ ]:
for name, d in [("baseline", base), ("fine-tuned", ft)]:
    print(f"{name:10s}: predictions mostly not in Telugu script = {100 * (d.telugu_share <= 0.5).mean():.0f}%")

## 7. Native-speaker labelling (your part)
This is the most valuable part of the project, and only a Telugu speaker can do it.

The next cell saves the **40 worst sentences** to `manual_error_labels.csv` in your Drive. Open it in Google Sheets, listen if needed, and fill the `category` column with one of:

`similar_sounds` · `vowel_length` · `aspiration` · `retroflex` · `foreign_name` · `english_word` · `number` · `word_joining` · `missing_words` · `hallucination` · `audio_quality` · `reference_error`

Add a short `note` in your own words. Save it as CSV in the same place, then run the cell after.

In [ ]:
label_file = f"{OUT}/manual_error_labels.csv"
if os.path.exists(label_file):
    print("Label file already exists, not overwriting:", label_file)
else:
    worst = ft.sort_values("cer", ascending=False).head(40)
    worst = worst[["id", "seconds", "reference", "prediction", "cer"]].assign(category="", note="")
    worst.to_csv(label_file, index=False)
    print("Saved. Now fill in the 'category' column:", label_file)

In [ ]:
labels = pd.read_csv(label_file, dtype={"category": str, "note": str}).fillna("")
done = labels[labels.category != ""]
if done.empty:
    print("No labels yet. Fill in the CSV first.")
else:
    counts = done.category.value_counts()
    print(f"{len(done)} sentences labelled")
    counts.plot.barh(figsize=(6, 3), title="Main cause of error (native-speaker labels)")
    plt.gca().invert_yaxis(); plt.tight_layout()
    plt.savefig(f"{OUT}/error_categories.png", dpi=150); plt.show()

## 8. Table for the README

In [ ]:
md = ["| Model | WER | CER |", "|---|---|---|"]
for _, r in results.iterrows():
    md.append(f"| {r['Model']} | {r['WER %']}% | {r['CER %']}% |")
print("\n".join(md))

## My notes

_Write 3–5 things **you** noticed here, in your own words. These notes become the analysis section of the README and your SOP._

1. 
2. 
3. 